# 🚀 AuthenticEye: End-to-End Colab Training Suite

This notebook automates the entire process: **Environment Setup** -> **Kaggle Dataset Download** -> **Data Preparation** -> **GPU Training**.

### Pre-requisites:
1. This notebook should be inside your `AuthenticEye` project folder on Google Drive.
2. Ensure you have your Kaggle API key ready.
3. Set Runtime to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

## 🛠️ Step 1: Mount Drive & Setup Environment

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Project Paths (Check if this matches your Drive folder name)
PROJECT_ROOT = '/content/drive/MyDrive/AuthenticEye'
AI_SERVICE_DIR = os.path.join(PROJECT_ROOT, 'ai-service')

if not os.path.exists(AI_SERVICE_DIR):
    print(f"❌ ERROR: Could not find project at {AI_SERVICE_DIR}")
    print("Please verify the folder name in your Google Drive.")
else:
    os.chdir(AI_SERVICE_DIR)
    print(f"✅ Working directory set to: {os.getcwd()}")

# 3. Install Requirements
print("📦 Installing dependencies...")
!pip install -q timm albumentations opencv-python-headless open_clip_torch datasets
!pip install -q -r requirements.txt

import torch
print(f"🔥 GPU STATUS: {'Enabled (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'DISABLED'}")

## 🔑 Step 2: Configure Kaggle Credentials
Run this cell to set up your Kaggle API so we can download the dataset automatically.

In [ ]:
import json
import os

# ENTER YOUR KAGGLE DETAILS HERE
KAGGLE_USERNAME = "" # Your Kaggle username
KAGGLE_KEY = ""      # Your Kaggle API Key

if KAGGLE_USERNAME and KAGGLE_KEY:
    kaggle_cred = {"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
        json.dump(kaggle_cred, f)
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("✅ Kaggle credentials configured!")
else:
    print("⚠️ Please enter your Kaggle credentials to enable automated downloads.")

## 📂 Step 3: Dataset Preparation
This will download the 140k Real/Fake dataset from Kaggle and partition it for training.

In [ ]:
# Create data directories on Drive to ensure persistence
!mkdir -p ./raw_data
!mkdir -p ./prepared_data

print("📥 Starting dataset download and preparation...")

# We limit to 500 images per class/split (total ~2000) for a faster startup.
# Change --limit to increase/decrease the dataset size.
!python scripts/prepare_dataset.py \
    --source ./raw_data \
    --dest ./prepared_data \
    --download-kaggle xhlulu/140k-real-and-fake-faces \
    --limit 500

## 🏋️ Step 4: Full Multi-Model Training
This section starts the training for the core models.

In [ ]:
# Select which model to run by uncommenting or adding more commands

print("🚀 Starting EfficientNet-B4 Training...")
!python training/trainer.py \
    --model efficientnet_b4 \
    --data_dir ./prepared_data \
    --epochs 50 \
    --batch_size 32 \
    --lr 1e-4 \
    --use_amp

# Optional: Uncomment to chain training
# print("🚀 Starting Vision Transformer Training...")
# !python training/trainer.py --model vit --data_dir ./prepared_data --epochs 30 --batch_size 16 --use_amp

## 📊 Step 5: Progress Monitoring

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../tensorboard_logs